In [1]:
!pip install segmentation-models-pytorch --quiet

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import cv2
import numpy as np
from sklearn.metrics import jaccard_score

print("All libraries installed and imported")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.5 MB/s eta 0:00:00
All libraries installed and imported


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
torch.cuda.empty_cache()

class PolypsDataset(Dataset):
    def __init__(self, image_names, image_dir, mask_dir):
        self.image_names = image_names
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    
    def __len__(self):
        return len(self.image_names)
    
    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = f"{self.image_dir}/{img_name}"
        mask_path = f"{self.mask_dir}/{img_name}"
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        image = cv2.resize(image, (128, 128))
        mask = cv2.resize(mask, (128, 128))
        
        image = image.astype(np.float32) / 255.0
        image = (image - self.mean) / self.std
        mask = (mask > 127).astype(np.float32)
        
        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()
        
        return image, mask

image_dir = '/kaggle/input/datasets/debeshjha1/kvasirseg/Kvasir-SEG/Kvasir-SEG/images'
mask_dir = '/kaggle/input/datasets/debeshjha1/kvasirseg/Kvasir-SEG/Kvasir-SEG/masks'

image_names = sorted(os.listdir(image_dir))
train_names, val_names = train_test_split(image_names, test_size=0.2, random_state=42)

train_dataset = PolypsDataset(train_names, image_dir, mask_dir)
val_dataset = PolypsDataset(val_names, image_dir, mask_dir)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Device: cuda
Train: 800, Val: 200


In [3]:
# Initialize with ResNet50 encoder
model = smp.Unet(
    encoder_name='resnet50',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None
).to(device)

print("ResNet50 U-Net initialized")

# Unfreeze encoder weights
for param in model.encoder.parameters():
    param.requires_grad = True

print("Encoder weights unfrozen (trainable)")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

ResNet50 U-Net initialized
Encoder weights unfrozen (trainable)


In [4]:
# Separate parameters: encoder vs decoder
encoder_params = list(model.encoder.parameters())
decoder_params = list(model.decoder.parameters()) + list(model.segmentation_head.parameters())

# Create optimizer with different learning rates
optimizer = torch.optim.Adam([
    {'params': encoder_params, 'lr': 1e-4},      # Slow fine-tuning
    {'params': decoder_params, 'lr': 1e-3}       # Fast learning
])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

print("Optimizer with differential LR initialized")

Optimizer with differential LR initialized


In [5]:
criterion_bce = nn.BCEWithLogitsLoss()

def dice_loss(pred, target):
    smooth = 1e-5
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    return 1 - (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def combined_loss(pred, target):
    return 0.5 * criterion_bce(pred, target) + 0.5 * dice_loss(pred, target)

In [6]:
num_epochs = 30
best_dice = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    # Validation
    model.eval()
    val_dice = 0.0
    val_loss = 0.0
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = combined_loss(outputs, masks)
            val_loss += loss.item()
            
            dice = 1 - dice_loss(outputs, masks)
            val_dice += dice.item()
    
    val_dice /= len(val_loader)
    val_loss /= len(val_loader)
    
    scheduler.step(val_dice)
    
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), 'resnet50_unfrozen_best.pth')
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")

print(f"\n✅ Best Dice: {best_dice:.4f}")

Epoch 5/30 | Train Loss: 0.1266 | Val Loss: 0.1903 | Val Dice: 0.8030
Epoch 10/30 | Train Loss: 0.0703 | Val Loss: 0.1705 | Val Dice: 0.8372
Epoch 15/30 | Train Loss: 0.0764 | Val Loss: 0.1701 | Val Dice: 0.8410
Epoch 20/30 | Train Loss: 0.0323 | Val Loss: 0.1701 | Val Dice: 0.8517
Epoch 25/30 | Train Loss: 0.0230 | Val Loss: 0.1744 | Val Dice: 0.8593
Epoch 30/30 | Train Loss: 0.0179 | Val Loss: 0.1766 | Val Dice: 0.8592

✅ Best Dice: 0.8598


In [7]:
model.load_state_dict(torch.load('resnet50_unfrozen_best.pth'))
model.eval()

dice_scores = []
iou_scores = []

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        outputs = model(images)
        pred = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
        mask = masks.cpu().numpy()
        
        for i in range(pred.shape[0]):
            pred_flat = pred[i].flatten().astype(int)
            mask_flat = mask[i].flatten().astype(int)
            
            dice = 1 - dice_loss(torch.tensor(pred[i]).unsqueeze(0), torch.tensor(mask[i]).unsqueeze(0)).item()
            iou = jaccard_score(mask_flat, pred_flat)
            
            dice_scores.append(dice)
            iou_scores.append(iou)

print(f"\n📊 ResNet50 Unfrozen Results:")
print(f"Dice: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"IoU:  {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")


📊 ResNet50 Unfrozen Results:
Dice: 0.2852 ± 0.1603
IoU:  0.7818 ± 0.2115
